# Multi-Output Regression Model: Regional Diet

This notebook trains a supervised multi-output regression model using `RandomForestRegressor`.

**Input feature:** 
- Country (categorical)

**Target variables:**
- Protein supply quantity (g/capita/day)
- Fat supply quantity (g/capita/day)
- Food supply (kcal/capita/day)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

### 1. Load the Dataset

In [2]:
# Load the dataset
file_path = "FoodBalanceSheets_E_All_Data.csv"
try:
    df = pd.read_csv(file_path, encoding='latin-1')
except Exception:
    # Fallback to the absolute path if run from outside the folder
    df = pd.read_csv("C:/Users/Abhinand Prameesh/Downloads/FoodBalanceSheets_E_All_Data.csv/FoodBalanceSheets_E_All_Data.csv", encoding='latin-1')

print("Data loaded successfully. Initial shape:", df.shape)

Data loaded successfully. Initial shape: (388181, 33)


### 2. Filter Rows by Element
Filtering for the required target parameters.

In [3]:
elements_to_keep = [
    'Protein supply quantity (g/capita/day)',
    'Fat supply quantity (g/capita/day)',
    'Food supply (kcal/capita/day)'
]

df_filtered = df[df['Element'].isin(elements_to_keep)]
print(f"Shape after filtering by elements: {df_filtered.shape}")

Shape after filtering by elements: (71745, 33)


### 3. Select the Latest Year Column

In [4]:
# Set the year explicitly to 2021
latest_year = 'Y2021'
print(f"Using year column: {latest_year}")

Using year column: Y2021


### 4. Reshape the Data
Pivot the dataset so each country has Protein, Fat, and Calories columns.

In [5]:
# Depending on dataset format, the country column can be 'Area' or 'Country'
country_col = 'Area' if 'Area' in df.columns else 'Country'

# Subset the dataframe before reshaping
df_subset = df_filtered[[country_col, 'Element', latest_year]]

# Pivot elements into columns per country
df_pivot = df_subset.pivot_table(index=country_col, columns='Element', values=latest_year, aggfunc='sum').reset_index()

# Rename columns according to target variables
df_pivot.rename(columns={
    'Fat supply quantity (g/capita/day)': 'Fat',
    'Food supply (kcal/capita/day)': 'Calories',
    'Protein supply quantity (g/capita/day)': 'Protein'
}, inplace=True)

# Drop rows with missing values
df_pivot = df_pivot.dropna()

print("Reshaped Data Head:")
print(df_pivot.head())

Reshaped Data Head:
Element         Area     Fat  Calories  Protein
0        Afghanistan  169.75   8791.90   242.74
1             Africa  228.44  10291.04   263.80
2            Albania  457.75  13594.90   480.51
3            Algeria  402.99  13985.25   374.90
4           Americas  535.59  13539.83   417.24


### 5. Encode the Country Column

In [6]:
le = LabelEncoder()
df_pivot['Country_Encoded'] = le.fit_transform(df_pivot[country_col])
print(df_pivot[[country_col, 'Country_Encoded']].head())

Element         Area  Country_Encoded
0        Afghanistan                0
1             Africa                1
2            Albania                2
3            Algeria                3
4           Americas                4


### 6 & 7. Split Dataset into Train/Test

In [7]:
# Feature variables
X = df_pivot[['Country_Encoded']]
# Target variables (Multiple outputs)
y = df_pivot[['Protein', 'Fat', 'Calories']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {X_train.shape[0]}, Testing instances: {X_test.shape[0]}")

Training instances: 176, Testing instances: 44


### 8. Train the RandomForestRegressor and Evaluate

In [8]:
# Train a RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict the targets for the test set
y_pred = rf_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("Model Evaluation Stats:")
print(f"R2 Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")

Model Evaluation Stats:
R2 Score: -0.1698
Mean Absolute Error (MAE): 667.2183


### 9. Save the Trained Model

In [9]:
model_filename = "regional_diet_model.pkl"
joblib.dump(rf_model, model_filename)
print(f"Model seamlessly exported as {model_filename}")

Model seamlessly exported as regional_diet_model.pkl
